# Lab 02: NUMPY FOR DATA SCIENCE
## Bài toán: Dự đoán khả năng có thay đổi công việc Khoa học dữ liệu

## Tiền xử lý dữ liệu

### Import thư viện và đọc dữ liệu


In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from src import data_processing as dp
import numpy as np
import math

In [3]:
file_path = '../data/raw/aug_train.csv'

data = np.genfromtxt(file_path, delimiter=',', dtype=str, skip_header=1)
headers = np.genfromtxt(file_path, delimiter=',', dtype=str, max_rows=1)

print("Headers:", headers)
print("Data mẫu:", data[:5])

Headers: ['enrollee_id' 'city' 'city_development_index' 'gender'
 'relevent_experience' 'enrolled_university' 'education_level'
 'major_discipline' 'experience' 'company_size' 'company_type'
 'last_new_job' 'training_hours' 'target']
Data mẫu: [['8949' 'city_103' '0.92' 'Male' 'Has relevent experience'
  'no_enrollment' 'Graduate' 'STEM' '>20' '' '' '1' '36' '1.0']
 ['29725' 'city_40' '0.7759999999999999' 'Male' 'No relevent experience'
  'no_enrollment' 'Graduate' 'STEM' '15' '50-99' 'Pvt Ltd' '>4' '47'
  '0.0']
 ['11561' 'city_21' '0.624' '' 'No relevent experience'
  'Full time course' 'Graduate' 'STEM' '5' '' '' 'never' '83' '0.0']
 ['33241' 'city_115' '0.789' '' 'No relevent experience' '' 'Graduate'
  'Business Degree' '<1' '' 'Pvt Ltd' 'never' '52' '1.0']
 ['666' 'city_162' '0.767' 'Male' 'Has relevent experience'
  'no_enrollment' 'Masters' 'STEM' '>20' '50-99' 'Funded Startup' '4' '8'
  '0.0']]


### Xử lý missing values
Phân thành 2 loại cột: numerical và categorical. Với numerical sẽ thay thế giá trị bị thiếu bằng giá trị median còn với categorical sẽ thay thế giá trị trống bằng 'Unknown'

In [4]:
# 1. Xác định index các cột
numerical_cols, categorical_cols, target_idx, exp_idx, job_idx = dp.identify_columns(headers)

# 2. Xử lý Numerical 
numerical_cleaned, numeric_col_indices = dp.process_numerical_data(data, numerical_cols, exp_idx, job_idx)

# 3. Xử lý Categorical
# Bước này trả về toàn bộ dữ liệu đã được làm sạch missing value
full_data_cleaned = dp.process_categorical_data(data, categorical_cols)

# QUAN TRỌNG: Chỉ giữ lại các cột Categorical để đưa vào bước Encode sau này
categorical_cleaned = full_data_cleaned[:, categorical_cols]

# 4. Lấy cột Target
target = data[:, target_idx].astype(float)

# In kết quả kiểm tra
print("Kích thước Numerical:", numerical_cleaned.shape)

Kích thước Numerical: (19158, 4)


### Phát hiện và xử lý outliers
Tìm các điểm dữ liệu bất thường quá thấp hoặc quá cao so với phần còn lại trong từng cột số. Không xóa outliers mà thay bằng giá trị giới hạn an toàn để giữ cấu trúc dữ liệu.

In [5]:
# Phát hiện và xử lý outliers (Winsorize)
numerical_winsorized = dp.handle_outliers(numerical_cleaned, headers, numeric_col_indices)

--- Phát hiện Outliers ---
Column city_development_index: 17 outliers
Column training_hours: 984 outliers


### Chuẩn hóa dữ liệu
Chuẩn hóa giá trị mỗi cột về khoảng [0, 1]

In [6]:
# Chuẩn hóa Min-Max (Vectorized)
min_max_normalized = dp.normalize_data(numerical_winsorized)

# Gán vào biến X_numeric để dùng cho các bước ghép nối (concat) phía sau
X_numeric = min_max_normalized

# Kiểm tra kết quả
print("Kích thước sau chuẩn hóa:", X_numeric.shape)

Kích thước sau chuẩn hóa: (19158, 4)


In [7]:
# Thực hiện Encoding và ghép toàn bộ features. Ordinal Encode và One-Hot Encode
X_final, final_headers = dp.encode_features(
    min_max_normalized,    # 
    categorical_cleaned,   
    headers,             
    numeric_col_indices,   
    categorical_cols      
)
# In kết quả kiểm tra
print(f"Đã hoàn tất! Kích thước Features (X_final): {X_final.shape}")

--- Đang thực hiện Encoding ---
Đã hoàn tất! Kích thước Features (X_final): (19158, 153)


### Lưu dữ liệu


In [10]:
output_file = '../data/processed/processed_data.csv'

# Lưu dữ liệu
dp.save_processed_data(X_final, target, final_headers, output_file)
print("XỬ LÝ HOÀN TẤT!")

Đã lưu file tại: ../data/processed/processed_data.csv
Kích thước dữ liệu: (19158, 154)
XỬ LÝ HOÀN TẤT!
